# Lab02 - Sơ đồ luồng logic Nhiệm vụ 2.1
**Nhóm 16 - NT101.R11.2**  
**Hà Nguyễn Hiếu An (24520043), Nguyễn Hữu Nghĩa (24521144)**

Sơ đồ dùng **Matplotlib** để vẽ khối/mũi tên, **Seaborn** để tạo bảng màu và heatmap Hamming. Gồm luồng chính, chi tiết bốn vòng Feistel và số liệu từ mã nguồn. Không đưa kiểm tra đầu vào, nhập liệu hay kiểm thử vào sơ đồ.

Hai lời gọi tuần tự cho M1 rồi M2 được gộp thành vòng lặp minh họa; không thay đổi thứ tự xử lý. Giải mã là bước bổ sung đang có trong chương trình. `compare_avalanche` gọi lại `feistel_trace` cho cả hai bản rõ.

Cài thư viện vẽ từ gốc repository: `python -m pip install -r Lab2/requirements-visualization.txt`. Chọn **Run All**; xuất PNG 300 DPI và SVG vào `figures/`. Notebook thuật toán không bị sửa.

Lấy đúng các hàm trong cell `feistel-core` của notebook Task 2.1, không chạy các cell ví dụ, nhập liệu hay kiểm thử.

In [1]:
import ast
import json
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Polygon
from IPython.display import display

# Hỗ trợ kernel chạy tại Lab2 hoặc tại gốc repository.
lab_dir = next((folder for folder in (Path.cwd(), Path.cwd() / 'Lab2')
                if (folder / 'Lab2_Task2.1.ipynb').is_file()), None)
if lab_dir is None:
    raise FileNotFoundError('Chạy notebook từ thư mục Lab2 hoặc gốc repository.')
source_path = lab_dir / 'Lab2_Task2.1.ipynb'
notebook = json.loads(source_path.read_text(encoding='utf-8'))
core_cell = next(cell for cell in notebook['cells'] if cell.get('id') == 'feistel-core')
core_source = core_cell['source']
core_source = ''.join(core_source) if isinstance(core_source, list) else core_source
# Chỉ nạp định nghĩa hàm của bài, không chạy ví dụ hay kiểm thử.
function_nodes = [node for node in ast.parse(core_source).body if isinstance(node, ast.FunctionDef)]
namespace = {}
exec(compile(ast.Module(body=function_nodes, type_ignores=[]), str(source_path), 'exec'), namespace)
key, messages = 0x12, (0xAB, 0xAC)
subkeys = namespace['generate_subkeys'](key)
traces = [namespace['feistel_trace'](message, key) for message in messages]
ciphertexts = [(states[-1][0] << 4) | states[-1][1] for states in traces]
distances = [((left1 ^ left2).bit_count() + (right1 ^ right2).bit_count())
             for (left1, right1), (left2, right2) in zip(*traces)]

sns.set_theme(style='white', context='notebook', font='DejaVu Sans')
plt.rcParams.update({'svg.fonttype': 'none', 'axes.unicode_minus': False})
font_manager.findfont('DejaVu Sans', fallback_to_default=False)
palette = sns.color_palette('colorblind', 5).as_hex()
colors = {'main': palette[0], 'round': palette[2], 'measure': palette[4],
          'optional': palette[3], 'ink': '#203247', 'muted': '#596a7c'}
print('Khóa con:', subkeys)
print('Bản mã:', [f'0x{value:02X}' for value in ciphertexts])
print('Hamming từ khởi tạo đến vòng 4:', distances)

Khóa con: [2, 1, 3, 4]
Bản mã: ['0x94', '0xE3']
Hamming từ khởi tạo đến vòng 4: [3, 6, 3, 3, 6]


In [ ]:
fig = plt.figure(figsize=(16, 21), dpi=120, facecolor='white')
plt.close(fig)  # Chỉ hiển thị hình hoàn chỉnh ở cell xuất ảnh.
ax = fig.add_axes([0, 0, 1, 1])
ax.set(xlim=(0, 1), ylim=(0, 1))
ax.axis('off')
node_bounds, node_texts = {}, []


def panel(x, y, width, height, color):
    patch = FancyBboxPatch((x, y), width, height,
                          boxstyle='round,pad=0.006,rounding_size=0.012',
                          facecolor=color, edgecolor=color, alpha=0.055, linewidth=1)
    ax.add_patch(patch)


def node(name, x, y, width, height, title, body='', color=None, shape='box', dashed=False):
    color = color or colors['main']
    bounds = (x - width / 2, y - height / 2, width, height)
    if shape == 'decision':
        patch = Polygon([(x, y + height / 2), (x + width / 2, y),
                         (x, y - height / 2), (x - width / 2, y)], closed=True,
                        facecolor='white', edgecolor=color, linewidth=1.7, zorder=3)
    else:
        rounding = 0.015 if shape == 'terminal' else 0.008
        patch = FancyBboxPatch(bounds[:2], width, height,
                              boxstyle=f'round,pad=0,rounding_size={rounding}',
                              facecolor='white', edgecolor=color, linewidth=1.7,
                              linestyle='--' if dashed else '-', zorder=3)
    ax.add_patch(patch)
    node_bounds[name] = bounds
    if body:
        title_artist = ax.text(x, y + height * 0.28, title, ha='center', va='center',
                               fontsize=14, weight='bold', color=color, zorder=4)
        body_artist = ax.text(x, y - height * 0.13, body, ha='center', va='center',
                              fontsize=12.6, linespacing=1.45, color=colors['ink'], zorder=4)
        node_texts.extend([(name, title_artist), (name, body_artist)])
    else:
        artist = ax.text(x, y, title, ha='center', va='center', fontsize=13.5,
                         weight='bold', linespacing=1.25, color=color, zorder=4)
        node_texts.append((name, artist))


def anchor(name, side):
    x, y, width, height = node_bounds[name]
    return {'top': (x + width / 2, y + height), 'bottom': (x + width / 2, y),
            'left': (x, y + height / 2), 'right': (x + width, y + height / 2)}[side]


def arrow(points, color=None, dashed=False):
    color = color or colors['ink']
    style = '--' if dashed else '-'
    if len(points) > 2:
        ax.plot([p[0] for p in points[:-1]], [p[1] for p in points[:-1]],
                color=color, linewidth=1.45, linestyle=style, zorder=2)
    ax.add_patch(FancyArrowPatch(points[-2], points[-1], arrowstyle='-|>',
                                mutation_scale=16, linewidth=1.45, color=color,
                                linestyle=style, shrinkA=0, shrinkB=2, zorder=2))


def label(x, y, text, rotation=0, color=None):
    ax.text(x, y, text, ha='center', va='center', rotation=rotation,
            fontsize=11.4, color=color or colors['ink'], zorder=5,
            bbox={'facecolor': 'white', 'edgecolor': 'none', 'pad': 2})


panel(0.025, 0.325, 0.472, 0.59, colors['main'])
panel(0.525, 0.325, 0.45, 0.59, colors['round'])
ax.text(0.5, 0.975, 'LUỒNG LOGIC THỰC HIỆN TASK 2.1', ha='center', va='center',
        fontsize=24, weight='bold', color=colors['ink'])
ax.text(0.5, 0.951, 'Feistel 8 bit • 4 vòng • Cùng khóa 0x12 • Quan sát sự lan truyền thay đổi',
        ha='center', fontsize=13, color=colors['muted'])
ax.text(0.26, 0.929, 'A. LUỒNG CHÍNH', ha='center', fontsize=15, weight='bold', color=colors['main'])
ax.text(0.75, 0.929, 'B. CHI TIẾT feistel_trace(msg, key)', ha='center', fontsize=15, weight='bold', color=colors['round'])

In [2]:
node('start', 0.255, 0.89, 0.20, 0.028, 'BẮT ĐẦU', shape='terminal')
node('input', 0.255, 0.834, 0.35, 0.064, '1. Khởi tạo dữ liệu',
     'M1 = 0xAB; M2 = 0xAC; key = 0x12\nM1 XOR M2 = 0x07: khác 3 bit')
node('select', 0.255, 0.758, 0.35, 0.061, '2. Chọn bản rõ cần xử lý',
     'Lần đầu: msg = M1\nLần tiếp theo: msg = M2, giữ nguyên khóa')
node('track', 0.255, 0.669, 0.35, 0.079, '3. track_avalanche(msg, key)',
     'Gọi feistel_trace (chi tiết ở cột B)\nIn trạng thái khởi tạo và sau 4 vòng\nTrả C = (L4 << 4) | R4')
node('decrypt', 0.255, 0.568, 0.35, 0.069, '4. Giải mã bổ sung',
     'feistel_decrypt(C, key)\nDuyệt khóa con ngược, khôi phục msg',
     color=colors['optional'], dashed=True)
node('more', 0.255, 0.475, 0.27, 0.067, 'Còn bản rõ\nchưa xử lý?', shape='decision')
node('compare', 0.255, 0.375, 0.35, 0.086, '5. compare_avalanche(M1, M2, key)',
     'feistel_trace cho M1 và M2 (tính lại)\nĐối chiếu (L, R) tại cùng chỉ số vòng\nĐếm bit khác bằng XOR và bit_count()', color=colors['measure'])

for source, target in [('start', 'input'), ('input', 'select'), ('select', 'track'),
                       ('track', 'decrypt'), ('decrypt', 'more'), ('more', 'compare')]:
    arrow([anchor(source, 'bottom'), anchor(target, 'top')])
label(0.29, 0.429, 'Không')
arrow([anchor('more', 'left'), (0.05, 0.475), (0.05, 0.758), anchor('select', 'left')])
label(0.05, 0.60, 'Có: chọn M2', rotation=90)
label(0.255, 0.612, 'Đã lưu đủ 5 trạng thái rồi mới in', color=colors['muted'])

NameError: name 'node' is not defined

In [ ]:
node('subkeys', 0.75, 0.859, 0.34, 0.079, 'generate_subkeys(key)',
     'K1 = key & 0x0F; K2 = (key >> 4) & 0x0F\nK3 = (key + 1) & 0x0F; K4 = (key + 2) & 0x0F\nkey = 0x12: ' + ', '.join(f'0x{value:X}' for value in subkeys), color=colors['round'])
node('split', 0.75, 0.762, 0.34, 0.078, 'Tách khối 8 bit và lưu trạng thái đầu',
     'L = (msg >> 4) & 0x0F; R = msg & 0x0F\nstates = [(L, R)]; i = 1\nM1: (1010, 1011); M2: (1010, 1100)', color=colors['round'])
node('condition', 0.75, 0.673, 0.20, 0.054, 'i ≤ 4?', color=colors['round'], shape='decision')
node('round', 0.75, 0.577, 0.34, 0.091, 'feistel_round(L, R, Ki)',
     'F(R, Ki) = (R XOR Ki) & 0x0F\nL_mới = R\nR_mới = (L XOR F(R, Ki)) & 0x0F', color=colors['round'])
node('save', 0.75, 0.474, 0.34, 0.066, 'Lưu kết quả vòng',
     'Cập nhật đồng thời L, R\nstates.append((L, R)); i = i + 1', color=colors['round'])
node('return', 0.75, 0.372, 0.34, 0.079, 'Trả danh sách states',
     '5 trạng thái: khởi tạo và 4 vòng\ntrack_avalanche in rồi ghép bản mã\nKhông đổi chỗ thêm sau vòng cuối', color=colors['round'])

for source, target in [('subkeys', 'split'), ('split', 'condition'),
                       ('condition', 'round'), ('round', 'save')]:
    arrow([anchor(source, 'bottom'), anchor(target, 'top')], color=colors['round'])
label(0.775, 0.636, 'Có', color=colors['round'])
arrow([anchor('save', 'left'), (0.547, 0.474), (0.547, 0.673), anchor('condition', 'left')], color=colors['round'])
label(0.547, 0.565, 'Vòng tiếp theo', rotation=90, color=colors['round'])
arrow([anchor('condition', 'right'), (0.95, 0.673), (0.95, 0.372), anchor('return', 'right')], color=colors['round'])
label(0.95, 0.543, 'Không: đủ 4 vòng', rotation=90, color=colors['round'])

# Nét đứt chỉ quan hệ gọi hàm, không phải nhánh thực thi mới.
arrow([anchor('track', 'right'), (0.511, 0.669), (0.511, 0.859), anchor('subkeys', 'left')],
      color=colors['main'], dashed=True)
label(0.511, 0.776, 'Gọi hàm', rotation=90, color=colors['main'])
ax.text(0.75, 0.312, 'i là chỉ số minh họa cho vòng lặp qua bốn khóa con.',
        ha='center', fontsize=11.5, color=colors['muted'])

In [ ]:
node('measure', 0.255, 0.26, 0.35, 0.078, '6. Đo chênh lệch từng vòng',
     'dL = bit_count(L1 XOR L2)\ndR = bit_count(R1 XOR R2)\nd = dL + dR; tỷ lệ = d / 8 × 100%', color=colors['measure'])
arrow([anchor('compare', 'bottom'), anchor('measure', 'top')], color=colors['measure'])

ax.text(0.75, 0.29, 'KHOẢNG CÁCH HAMMING THỰC TẾ', ha='center', fontsize=13.5,
        weight='bold', color=colors['measure'])
heat_ax = fig.add_axes([0.56, 0.233, 0.37, 0.038])
sns.heatmap([distances], ax=heat_ax, annot=True, fmt='d', cbar=False,
            cmap=sns.light_palette(colors['measure'], as_cmap=True), vmin=0, vmax=8,
            linewidths=1.5, linecolor='white', yticklabels=False,
            xticklabels=['Khởi tạo', 'Vòng 1', 'Vòng 2', 'Vòng 3', 'Vòng 4'],
            annot_kws={'size': 17, 'weight': 'bold'})
heat_ax.tick_params(axis='x', labelsize=12, length=0, pad=8, labelrotation=0)
ax.text(0.745, 0.193, 'C1 = 0x' + f'{ciphertexts[0]:02X}' + '     |     C2 = 0x' + f'{ciphertexts[1]:02X}'
        + f'     |     Khác {distances[-1]}/8 bit = {distances[-1] / 8:.0%}',
        ha='center', fontsize=13, weight='bold', color=colors['ink'])

node('conclusion', 0.255, 0.159, 0.35, 0.079, '7. Nhận xét sự lan truyền thay đổi',
     'Khác biệt lan sang cả hai nửa\nSố bit khác nhau không tăng đều\nF chỉ XOR, không trộn các vị trí bit', color=colors['measure'])
node('end', 0.255, 0.09, 0.20, 0.028, 'KẾT THÚC', shape='terminal')
arrow([anchor('measure', 'bottom'), anchor('conclusion', 'top')], color=colors['measure'])
arrow([anchor('conclusion', 'bottom'), anchor('end', 'top')])

ax.text(0.745, 0.151, 'Lưu ý khi viết report', ha='center', fontsize=13.5, weight='bold', color=colors['ink'])
ax.text(0.745, 0.111, 'Đề ghi khác 1 bit, thực tế khác 3 bit.\n75% ở đầu ra không đủ chứng minh avalanche mạnh.\nNét đứt: gọi hàm hoặc bước giải mã bổ sung.',
        ha='center', va='center', fontsize=12.5, linespacing=1.65, color=colors['muted'])
ax.text(0.5, 0.046, 'Luồng chính giữ thứ tự: xử lý M1, xử lý M2, rồi so sánh. Không hiển thị kiểm thử hay kiểm tra đầu vào.',
        ha='center', fontsize=11.8, color=colors['muted'])
ax.text(0.5, 0.026, 'Matplotlib: khối và mũi tên  |  Seaborn: bảng màu và heatmap  |  Nguồn số liệu: cell feistel-core',
        ha='center', fontsize=11.8, color=colors['muted'])

In [ ]:
output_dir = lab_dir / 'figures'
output_dir.mkdir(parents=True, exist_ok=True)
output_png = output_dir / 'Lab2_Task2.1_Flowchart.png'
output_svg = output_dir / 'Lab2_Task2.1_Flowchart.svg'
fig.savefig(output_png, dpi=300, facecolor='white')
fig.savefig(output_svg, facecolor='white')
print('PNG 300 DPI:', output_png.resolve())
print('SVG:', output_svg.resolve())
display(fig)
plt.close(fig)

## Cách dùng trong report
- PNG 300 DPI: chèn trực tiếp vào Word. SVG: phóng to không vỡ nét.
- Chú thích đề xuất: **Sơ đồ luồng xử lý Feistel 8 bit và so sánh sự lan truyền thay đổi qua bốn vòng.**
- Hai bản rõ đề bài khác 3 bit, không phải 1 bit. Khoảng cách Hamming không tăng đều; không kết luận mô hình đạt hiệu ứng thác đổ mạnh.
- Nguồn dữ liệu: `Lab2_Task2.1.ipynb`, cell `feistel-core`.
- Thư viện: https://matplotlib.org/stable/api/patches_api.html; https://seaborn.pydata.org/generated/seaborn.heatmap.html.